In [39]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("../data/clean_data/clean_dataset.csv")

In [3]:
df.columns.tolist()

['show_id',
 'type',
 'title',
 'director',
 'country',
 'date_added',
 'release_year',
 'rating',
 'duration',
 'listed_in']

In [18]:
feature_df = df[["title","type","director","country","listed_in","release_year","rating"]]
feature_df

,title,type,director,country,listed_in,release_year,rating
0,Dick Johnson Is Dead,Movie,Kirsten Johnson,United States,Documentaries,2020,PG-13
1,Ganglands,TV Show,Julien Leclercq,France,"Crime TV Shows, International TV Shows, TV Act...",2021,TV-MA
2,Midnight Mass,TV Show,Mike Flanagan,United States,"TV Dramas, TV Horror, TV Mysteries",2021,TV-MA
3,Confessions of an Invisible Girl,Movie,Bruno Garotti,Brazil,"Children & Family Movies, Comedies",2021,TV-PG
4,Sankofa,Movie,Haile Gerima,United States,"Dramas, Independent Movies, International Movies",1993,TV-MA
...,...,...,...,...,...,...,...
8785,Yunus Emre,TV Show,Not Given,Turkey,"International TV Shows, TV Dramas",2016,TV-PG
8786,Zak Storm,TV Show,Not Given,United States,Kids' TV,2016,TV-Y7
8787,Zindagi Gulzar Hai,TV Show,Not Given,Pakistan,"International TV Shows, Romantic TV Shows, TV ...",2012,TV-PG
8788,Yoko,TV Show,Not Given,Pakistan,Kids' TV,2016,TV-Y


In [6]:
feature_df.isnull().sum()

title           0
type            0
director        0
country         0
listed_in       0
release_year    0
rating          0
dtype: int64

In [7]:
feature_df.duplicated().sum()

np.int64(3)

In [8]:
feature_df.duplicated(subset="title").sum()

np.int64(3)

In [9]:
feature_df[
    feature_df.duplicated(subset="title",keep=False)
].sort_values("title")

,title,type,director,country,listed_in,release_year,rating
2925,15-Aug,Movie,Swapnaneel Jayakar,India,"Comedies, Dramas, Independent Movies",2019,TV-14
4261,15-Aug,Movie,Swapnaneel Jayakar,India,"Comedies, Dramas, Independent Movies",2019,TV-14
3285,22-Jul,Movie,Paul Greengrass,Norway,"Dramas, Thrillers",2018,R
4260,22-Jul,Movie,Paul Greengrass,Norway,"Dramas, Thrillers",2018,R
393,9-Feb,TV Show,Not Given,Pakistan,"International TV Shows, TV Dramas",2018,TV-14
537,9-Feb,TV Show,Not Given,Pakistan,"International TV Shows, TV Dramas",2018,TV-14


In [10]:
df[df["title"].isin(["15-Aug","22-jul","9-Feb"])]

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
393,s3997,TV Show,9-Feb,Not Given,Pakistan,2019-03-20,2018,TV-14,1 Season,"International TV Shows, TV Dramas"
537,s5965,TV Show,9-Feb,Not Given,Pakistan,2019-03-20,2018,TV-14,1 Season,"International TV Shows, TV Dramas"
2925,s3963,Movie,15-Aug,Swapnaneel Jayakar,India,2019-03-29,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"
4261,s5967,Movie,15-Aug,Swapnaneel Jayakar,India,2019-03-29,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"


In [11]:
df[df["show_id"].isin(["s3997","s5965","s3963","s5967"])]

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
393,s3997,TV Show,9-Feb,Not Given,Pakistan,2019-03-20,2018,TV-14,1 Season,"International TV Shows, TV Dramas"
537,s5965,TV Show,9-Feb,Not Given,Pakistan,2019-03-20,2018,TV-14,1 Season,"International TV Shows, TV Dramas"
2925,s3963,Movie,15-Aug,Swapnaneel Jayakar,India,2019-03-29,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"
4261,s5967,Movie,15-Aug,Swapnaneel Jayakar,India,2019-03-29,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"


In [13]:
df[df["title"] == "22-Jul"]

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
3285,s4523,Movie,22-Jul,Paul Greengrass,Norway,2018-10-10,2018,R,144 min,"Dramas, Thrillers"
4260,s5966,Movie,22-Jul,Paul Greengrass,Norway,2018-10-10,2018,R,144 min,"Dramas, Thrillers"


In [21]:
recommendation_df = df[[
    "show_id","title","type","director","country",
    "listed_in","release_year","rating"
]].drop_duplicates(subset="show_id",keep="first")

recommendation_df

,show_id,title,type,director,country,listed_in,release_year,rating
0,s1,Dick Johnson Is Dead,Movie,Kirsten Johnson,United States,Documentaries,2020,PG-13
1,s3,Ganglands,TV Show,Julien Leclercq,France,"Crime TV Shows, International TV Shows, TV Act...",2021,TV-MA
2,s6,Midnight Mass,TV Show,Mike Flanagan,United States,"TV Dramas, TV Horror, TV Mysteries",2021,TV-MA
3,s14,Confessions of an Invisible Girl,Movie,Bruno Garotti,Brazil,"Children & Family Movies, Comedies",2021,TV-PG
4,s8,Sankofa,Movie,Haile Gerima,United States,"Dramas, Independent Movies, International Movies",1993,TV-MA
...,...,...,...,...,...,...,...,...
8785,s8797,Yunus Emre,TV Show,Not Given,Turkey,"International TV Shows, TV Dramas",2016,TV-PG
8786,s8798,Zak Storm,TV Show,Not Given,United States,Kids' TV,2016,TV-Y7
8787,s8801,Zindagi Gulzar Hai,TV Show,Not Given,Pakistan,"International TV Shows, Romantic TV Shows, TV ...",2012,TV-PG
8788,s8784,Yoko,TV Show,Not Given,Pakistan,Kids' TV,2016,TV-Y


In [22]:
recommendation_df.duplicated().sum()

np.int64(0)

In [23]:
recommendation_df.isnull().sum()

show_id         0
title           0
type            0
director        0
country         0
listed_in       0
release_year    0
rating          0
dtype: int64

In [24]:
text_columns = ["director","country","listed_in"]

for col in text_columns:
    recommendation_df[col] = (
        recommendation_df[col].str.strip()
        .str.lower()
    )

In [25]:
recommendation_df[text_columns].head()

,director,country,listed_in
0,kirsten johnson,united states,documentaries
1,julien leclercq,france,"crime tv shows, international tv shows, tv act..."
2,mike flanagan,united states,"tv dramas, tv horror, tv mysteries"
3,bruno garotti,brazil,"children & family movies, comedies"
4,haile gerima,united states,"dramas, independent movies, international movies"


In [26]:
recommendation_df[text_columns] = (
    recommendation_df[text_columns]
    .replace("not given","")
)

In [27]:
recommendation_df[text_columns].head()

,director,country,listed_in
0,kirsten johnson,united states,documentaries
1,julien leclercq,france,"crime tv shows, international tv shows, tv act..."
2,mike flanagan,united states,"tv dramas, tv horror, tv mysteries"
3,bruno garotti,brazil,"children & family movies, comedies"
4,haile gerima,united states,"dramas, independent movies, international movies"


In [28]:
(recommendation_df[text_columns] == "").sum()

director     2588
country       287
listed_in       0
dtype: int64

In [30]:
recommendation_df[[
    "type","release_year","rating"
]].isnull().sum()

type            0
release_year    0
rating          0
dtype: int64

In [31]:
recommendation_df["combined_features"] = (
    recommendation_df["type"].astype(str)+ " " +
    recommendation_df["director"].astype(str)+ " " +
    recommendation_df["country"].astype(str)+ " "+
    recommendation_df["listed_in"].astype(str)+" "+
    recommendation_df["release_year"].astype(str)+" "+
    recommendation_df["rating"].astype(str)
)

In [32]:
recommendation_df[[
    "title","combined_features"
]]

,title,combined_features
0,Dick Johnson Is Dead,Movie kirsten johnson united states documentar...
1,Ganglands,"TV Show julien leclercq france crime tv shows,..."
2,Midnight Mass,"TV Show mike flanagan united states tv dramas,..."
3,Confessions of an Invisible Girl,Movie bruno garotti brazil children & family m...
4,Sankofa,"Movie haile gerima united states dramas, indep..."
...,...,...
8785,Yunus Emre,"TV Show turkey international tv shows, tv dra..."
8786,Zak Storm,TV Show united states kids' tv 2016 TV-Y7
8787,Zindagi Gulzar Hai,"TV Show pakistan international tv shows, roma..."
8788,Yoko,TV Show pakistan kids' tv 2016 TV-Y


In [36]:
tfidf = TfidfVectorizer(stop_words="english")

In [37]:
tfidf_matrix = tfidf.fit_transform(
    recommendation_df["combined_features"]
)

In [38]:
tfidf_matrix.shape

(8790, 6642)

In [40]:
similarity_matrix = cosine_similarity(tfidf_matrix)

In [41]:
similarity_matrix.shape

(8790, 8790)

In [42]:
selected_title = "Kota Factory"

title_index = recommendation_df[
    recommendation_df["title"].str.lower() == selected_title.lower()
].index[0]

In [43]:
title_index

6572

In [44]:
recommendation_df.loc[title_index,"title"]

'Kota Factory'

In [47]:
similarity_scores = list(enumerate(similarity_matrix[title_index]))

sorted_scores = sorted(
    similarity_scores,
    key=lambda x: x[1],
    reverse=True
)

top5 = sorted_scores[1:6]
recommended_indices = [index for index, score in top5]
recommendations = recommendation_df.iloc[recommended_indices][[
    "title","type","listed_in"
]].copy()

recommendations["similarity_score"] = [
    score for index,score in top5
]

recommendations

,title,type,listed_in,similarity_score
7552,College Romance,TV Show,"international tv shows, romantic tv shows, tv ...",0.894024
7487,Little Things,TV Show,"international tv shows, romantic tv shows, tv ...",0.891247
6964,Bhaag Beanie Bhaag,TV Show,"international tv shows, romantic tv shows, tv ...",0.889181
6987,Mismatched,TV Show,"international tv shows, romantic tv shows, tv ...",0.889181
7354,Taj Mahal 1989,TV Show,"international tv shows, romantic tv shows, tv ...",0.889181


In [48]:
recommendation_df.loc[
    recommendation_df["title"].str.lower() == "kota factory",
    ["title","type","country","listed_in","release_year","rating",'combined_features']
]

,title,type,country,listed_in,release_year,rating,combined_features
6572,Kota Factory,TV Show,india,"international tv shows, romantic tv shows, tv ...",2021,TV-MA,"TV Show india international tv shows, romanti..."


In [50]:
df.loc[
    df["title"] == "Kota Factory",
    ["title","listed_in"]
]

,title,listed_in
6572,Kota Factory,"International TV Shows, Romantic TV Shows, TV ..."


In [51]:
recommendation_df[
    recommendation_df["title"].isin([
        "Kota Factory",
        "College Romance",
        "Little Things",
        "Bhaag Beanie Bhaag",
        "Mismatched",
        "Taj Mahal 19899"
    ])
][["title","country","listed_in","release_year","rating"]]

,title,country,listed_in,release_year,rating
6572,Kota Factory,india,"international tv shows, romantic tv shows, tv ...",2021,TV-MA
6964,Bhaag Beanie Bhaag,india,"international tv shows, romantic tv shows, tv ...",2020,TV-MA
6987,Mismatched,india,"international tv shows, romantic tv shows, tv ...",2020,TV-MA
7487,Little Things,india,"international tv shows, romantic tv shows, tv ...",2019,TV-MA
7552,College Romance,india,"international tv shows, romantic tv shows, tv ...",2018,TV-MA


In [52]:
recommendation_df.iloc[recommended_indices]["title"].str.lower().eq(
    selected_title.lower()
).any()

np.False_

In [53]:
recommendations["title"].duplicated().sum()

np.int64(0)

In [54]:
recommendations

,title,type,listed_in,similarity_score
7552,College Romance,TV Show,"international tv shows, romantic tv shows, tv ...",0.894024
7487,Little Things,TV Show,"international tv shows, romantic tv shows, tv ...",0.891247
6964,Bhaag Beanie Bhaag,TV Show,"international tv shows, romantic tv shows, tv ...",0.889181
6987,Mismatched,TV Show,"international tv shows, romantic tv shows, tv ...",0.889181
7354,Taj Mahal 1989,TV Show,"international tv shows, romantic tv shows, tv ...",0.889181


In [55]:
selected_title = "Stranger Things"

title_index = recommendation_df[
    recommendation_df["title"].str.lower() == selected_title.lower()
].index[0]

In [56]:
similarity_scores = list(enumerate(similarity_matrix[title_index]))

sorted_scores = sorted(
    similarity_scores,
    key=lambda x: x[1],
    reverse=True
)

top5 = sorted_scores[1:6]
recommended_indices = [index for index, score in top5]
recommendations = recommendation_df.iloc[recommended_indices][[
    "title","type","listed_in"
]].copy()

recommendations["similarity_score"] = [
    score for index,score in top5
]

recommendations

,title,type,listed_in,similarity_score
6923,Chilling Adventures of Sabrina,TV Show,"tv horror, tv mysteries, tv sci-fi & fantasy",0.937293
7446,Nightflyers,TV Show,"tv horror, tv mysteries, tv sci-fi & fantasy",0.909681
7751,The OA,TV Show,"tv dramas, tv mysteries, tv sci-fi & fantasy",0.903854
8485,Helix,TV Show,"tv horror, tv mysteries, tv sci-fi & fantasy",0.895906
8122,The Vampire Diaries,TV Show,"tv dramas, tv mysteries, tv sci-fi & fantasy",0.869631


Evaluation of Recommendation Quality

The recommendation system was tested using two sample titles: Kota Factory and Stranger Things.

Evaluation checks:

1. The selected title was successfully excluded from its own recommendations.
2. No duplicate recommendations were generated.
3. The system generated five recommendations for each selected title.
4. Recommended titles shared relevant content features such as type, country, and genres.

Observations:

- Kota Factory received recommendations with similar metadata and categories.
- Stranger Things received recommendations related to horror, mystery, and science fiction.
- Cosine similarity scores helped rank titles according to their content-feature similarity.

Limitations:

- The model uses metadata rather than actual plot descriptions or user preferences.
- Similarity scores do not represent recommendation accuracy or user satisfaction.
- Further evaluation using user feedback or a ground-truth dataset would improve reliability.

Conclusion: The content-based recommendation system successfully generated and ranked recommendations using TF-IDF and cosine similarity.

Conclusion and Key Findings

In this project, a content-based recommendation system was developed using the Netflix dataset.

Key steps performed:

- Selected relevant content features such as type, director, country, genre, release year, and rating.
- Preprocessed text data by converting text to lowercase and handling placeholder values.
- Combined selected features into a single text column.
- Applied TF-IDF Vectorization to convert text into numerical vectors.
- Used Cosine Similarity to calculate similarity between Netflix titles.
- Generated the top 5 recommendations for selected titles.

Key Findings:

- The system generated relevant recommendations for Kota Factory and Stranger Things based on their available metadata.
- TF-IDF and Cosine Similarity helped rank similar titles.
- The system successfully excluded the selected title and avoided duplicate recommendations.

Final Outcome:

The project demonstrates how content-based filtering can recommend similar Netflix titles using metadata and text similarity techniques. Future improvements could include plot descriptions, cast information, user feedback, and more advanced recommendation algorithms.